In [2]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx"   # change to your file
df = pd.read_excel(file_path)

# Clean column names
df.columns = df.columns.str.strip()

# ===============================
# BASIC CALCULATIONS
# ===============================

# Reject %
df["Reject_%"] = np.where(
    df["Prodn"] > 0,
    (df["Rej"] / df["Prodn"]) * 100,
    0
)

# Quality Yield
df["Yield_%"] = np.where(
    df["Prodn"] > 0,
    (df["Ok"] / df["Prodn"]) * 100,
    0
)

# Cycle deviation %
df["Cycle_Deviation_%"] = np.where(
    df["CycleTime Tgt"] > 0,
    ((df["CycleTime Act"] - df["CycleTime Tgt"]) / df["CycleTime Tgt"]) * 100,
    0
)

# Cavity utilization %
df["Cavity_Utilization_%"] = np.where(
    df["Planned Cavity"] > 0,
    (df["Actual Cavity"] / df["Planned Cavity"]) * 100,
    0
)

# Availability indicator
df["Total_Loss_Time"] = (
    df["Downtime"] +
    df["CO Time"] +
    df["Break Time"]
)

# ===============================
# SUMMARY VIEW
# ===============================

summary_cols = [
    "Part Number",
    "Ok",
    "Rej",
    "Prodn",
    "Reject_%",
    "Yield_%",
    "Cycle_Deviation_%",
    "Cavity_Utilization_%",
    "Total_Loss_Time",
    "PE"
]

summary_df = df[summary_cols]

# ===============================
# SAVE
# ===============================

summary_df.to_excel("Production_17Feb_Diagnostic.xlsx", index=False)

print("Production diagnostic completed.")

Production diagnostic completed.


In [3]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx"   # change path if needed
df = pd.read_excel(file_path)

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# ===============================
# CHECK REQUIRED COLUMNS
# ===============================

required_cols = [
    "Part Number",
    "Prodn",
    "Run Time",
    "CycleTime Act",
    "Actual Cavity",
    "Pdt Time"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    print("Missing columns:", missing)
    print("Check column names in file.")
else:

    print("\n===== STEP 1: CAVITY TEST =====")

    # Theoretical production assuming cavity = running cavities
    df["expected_prodn"] = (
        (df["Run Time"] / df["CycleTime Act"]) * df["Actual Cavity"]
    )

    df["prodn_diff"] = df["Prodn"] - df["expected_prodn"]

    print(df[["Part Number", "Prodn", "expected_prodn", "prodn_diff"]].head(10))

    avg_diff = df["prodn_diff"].abs().mean()
    print("\nAverage production difference:", avg_diff)

    if avg_diff < df["Prodn"].mean() * 0.1:
        print("👉 Actual cavity likely represents running cavities.")
    else:
        print("👉 Actual cavity likely represents tool capacity or something else.")

    print("\n===== STEP 2: RUN TIME TEST =====")

    corr_runtime = df[["Run Time", "Prodn"]].corr().iloc[0,1]
    print("Correlation between Run Time and Production:", corr_runtime)

    if corr_runtime > 0.7:
        print("👉 Run time likely represents actual machine operating time.")
    else:
        print("👉 Run time may not be pure operating time — needs clarification.")

    print("\n===== STEP 3: PDT TIME TEST =====")

    corr_pdt = df[["Pdt Time", "Run Time"]].corr().iloc[0,1]
    print("Correlation between PDT Time and Run Time:", corr_pdt)

    if corr_pdt < 0:
        print("👉 PDT likely represents planned downtime.")
    else:
        print("👉 PDT meaning unclear — may not directly reduce run time.")

# ===============================
# SAVE RESULTS
# ===============================

df.to_excel("production_definition_test_results.xlsx", index=False)

print("\nDiagnostic results saved.")

Missing columns: ['Part Number', 'Prodn', 'Run Time', 'CycleTime Act', 'Actual Cavity', 'Pdt Time']
Check column names in file.

Diagnostic results saved.


In [6]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx"
df = pd.read_excel(file_path)

# Clean column names (remove extra spaces only)
df.columns = df.columns.str.strip()

# ===============================
# VERIFY REQUIRED COLUMNS
# ===============================

required_cols = [
    "Part Number",
    "Prodn",
    "Run Time",
    "CycleTime Act",
    "Actual Cavity",
    "Pdt Time"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    print("❌ Missing columns:", missing)
    print("👉 Run this to inspect headers:")
    print(df.columns.tolist())

else:

    print("\n===== STEP 1: CAVITY TEST =====")

    # Theoretical production assuming cavity = running cavities
    df["Expected Prodn"] = (
        (df["Run Time"] / df["CycleTime Act"]) * df["Actual Cavity"]
    )

    df["Prodn Diff"] = df["Prodn"] - df["Expected Prodn"]

    print(df[[
        "Part Number",
        "Prodn",
        "Expected Prodn",
        "Prodn Diff"
    ]].head(10))

    avg_diff = df["Prodn Diff"].abs().mean()
    print("\nAverage production difference:", avg_diff)

    if avg_diff < df["Prodn"].mean() * 0.1:
        print("👉 Actual Cavity likely represents RUNNING cavities.")
    else:
        print("👉 Actual Cavity likely represents TOOL capacity or needs clarification.")

    print("\n===== STEP 2: RUN TIME TEST =====")

    corr_runtime = df[["Run Time", "Prodn"]].corr().iloc[0,1]
    print("Correlation between Run Time and Production:", corr_runtime)

    if corr_runtime > 0.7:
        print("👉 Run Time likely represents actual machine operating time.")
    else:
        print("👉 Run Time meaning unclear — may include other factors.")

    print("\n===== STEP 3: PDT TIME TEST =====")

    corr_pdt = df[["Pdt Time", "Run Time"]].corr().iloc[0,1]
    print("Correlation between PDT Time and Run Time:", corr_pdt)

    if corr_pdt < 0:
        print("👉 PDT Time likely represents planned downtime.")
    else:
        print("👉 PDT Time meaning unclear — may not directly reduce runtime.")

# ===============================
# SAVE RESULTS
# ===============================

df.to_excel("production_definition_test_results.xlsx", index=False)

print("\n✅ Diagnostic results saved to file.")



===== STEP 1: CAVITY TEST =====
          Part Number  Prodn  Expected Prodn    Prodn Diff
0       S22127-007A0X  19944     2911.621053  17032.378947
1       S22127-003A0X    696       28.743590    667.256410
2  14SW330134-00005X0   9112      759.091892   8352.908108
3  14MA410307-00003X2    566        9.240924    556.759076
4       S11436-002A0X   1588       85.810169   1502.189831
5       S31741-010A0X   6936      483.445455   6452.554545
6       S31121-012A0X   1386       25.412757   1360.587243
7       S03058-002A0X   1814       62.785526   1751.214474
8       S12104-006A0X    938       13.135845    924.864155
9  14SW220197-00010X0   1320       51.078481   1268.921519

Average production difference: 2344.695951785812
👉 Actual Cavity likely represents TOOL capacity or needs clarification.

===== STEP 2: RUN TIME TEST =====
Correlation between Run Time and Production: 0.3850471637678222
👉 Run Time meaning unclear — may include other factors.

===== STEP 3: PDT TIME TEST =====
Correl